Собираем изначальные данные из https://datadryad.org/dataset/doi:10.5061/dryad.2jm63xsrf

In [ ]:
from google.colab import files
import zipfile
import os

In [ ]:
uploaded = files.upload()

uploaded_filename = list(uploaded.keys())[0]
print(f"загружен {uploaded_filename}")
with zipfile.ZipFile(uploaded_filename, "r") as z:
  z.extractall("data")
os.listdir("data")

Saving trees_dataset.zip to trees_dataset.zip
Загружен файл: trees_dataset.zip


['Phoenix_Final_2021-09-22 (1).csv',
 'Plano_Final_2021-09-22.csv',
 'Buffalo_Final_2021-09-22.csv',
 'Denver_Final_2021-09-22.csv',
 'Irvine_Final_2021-09-22.csv',
 'Columbus_Final_2021-09-22.csv',
 'Anaheim_Final_2021-09-22.csv',
 'Louisville_Final_2021-09-22.csv',
 'OklahomaCity_Final_2021-09-22.csv',
 'Madison_Final_2021-09-22.csv',
 'Phoenix_Final_2021-09-22.csv',
 'Miami_Final_2021-09-22.csv',
 'HuntingtonBeach_Final_2021-09-22.csv',
 'Baltimore_Final_2021-09-22.csv',
 'Greensboro_Final_2021-09-22.csv',
 'GardenGrove_Final_2021-09-22.csv',
 'Portland_Final_2021-09-22.csv',
 'Indianapolis_Final_2021-09-22.csv',
 'OverlandPark_Final_2021-09-22.csv',
 'Nashville_Final_2021-09-22.csv',
 'Albuquerque_Final_2021-09-22.csv',
 'Orlando_Final_2021-09-22.csv',
 'Milwaukee_Final_2021-09-22.csv',
 'Boston_Final_2021-09-22.csv',
 'GrandRapids_Final_2021-09-22.csv',
 'Houston_Final_2021-09-22.csv',
 'DesMoines_Final_2021-09-22.csv',
 'Durham_Final_2021-09-22.csv',
 'Atlanta_Final_2021-09-22.cs

In [ ]:
import pandas as pd
import glob
import os

In [ ]:
path = 'data'
all_files = glob.glob(os.path.join(path, "*.csv"))

li = []
for filename in all_files:
  df_city = pd.read_csv(filename, index_col=None, header=0)
  li.append(df_city)
df_trees = pd.concat(li, axis=0, ignore_index=True)
print(f"итог датасет {len(df_trees)}")

/tmp/ipykernel_2683/1918352711.py:13: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_city = pd.read_csv(filename, index_col=None, header=0)
/tmp/ipykernel_2683/1918352711.py:13: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_city = pd.read_csv(filename, index_col=None, header=0)
/tmp/ipykernel_2683/1918352711.py:13: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_city = pd.read_csv(filename, index_col=None, header=0)


Объединенный датасет готов. Строк: 2837059


In [ ]:
def sl(row):
  if pd.notna(row['retired_date']) and row['condition'] == 'Dead':
    return 'dead_confirmed'
  elif pd.notna(row['retired_date']):
    return 'removed_other'
  elif row['condition'] == 'Dead':
    return 'dead_confirmed'
  elif row['condition'] in ['Good', 'Fair']:
    return 'alive_healthy'
  elif row['condition'] == 'Poor':
    return 'alive_stressed'
  else:
    return 'unknown'
df_final['survival_label'] = df_final.apply(sl, axis=1)

In [ ]:
df_final.shape

(2837080, 40)

In [ ]:
cols = df_final.select_dtypes('object').columns

for c in cols:
  df_final[c] = df_final[c].astype('string')
df_final.to_parquet('/content/drive/MyDrive/df_final.parquet')
print('ок')

NameError: name 'df_final' is not defined

Taking out the file

In [ ]:
import pandas as pd

In [ ]:
f = '/content/drive/MyDrive/df_final.parquet'
df = pd.read_parquet(f)

In [ ]:
df.shape

(2837080, 40)

In [ ]:
import pandas as pd
import requests
import time

Weather data

https://archive-api.open-meteo.com


In [ ]:
df.columns

Index(['common_name', 'scientific_name', 'city', 'state',
       'diameter_breast_height_CM', 'height_M', 'native', 'height_binned_M',
       'diameter_breast_height_binned_CM', 'greater_metro', 'city_ID',
       'tree_ID', 'planted_date', 'most_recent_observation', 'retired_date',
       'most_recent_observation_type', 'longitude_coordinate',
       'latitude_coordinate', 'location_type', 'zipcode', 'address',
       'neighborhood', 'location_name', 'ward', 'district', 'overhead_utility',
       'condition', 'percent_population', 'column_name', 'definition',
       'sci_name_clean', 'symbol', 'Synonym Symbol',
       'Scientific Name with Author', 'Common Name', 'Family', 'duration',
       'wetland_indicator', 'growth_habit', 'survival_label'],
      dtype='object')

In [ ]:
s=df[df['latitude_coordinate'].notna()]
s=s.groupby('state')[['latitude_coordinate', 'longitude_coordinate']].mean()
s=s.reset_index()

s.columns = ['state', 'lat', 'lon']
print(len(s))

19


In [ ]:
def get_data(lat, lon):
  u = 'https://archive-api.open-meteo.com/v1/archive'
  p = {
    'latitude': lat,
    'longitude': lon,
    'start_date': '2015-01-01',
    'end_date': '2022-12-31',
    'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum,et0_fao_evapotranspiration',
    'hourly': 'soil_moisture_0_to_7cm,soil_temperature_0_to_7cm',
    'timezone': 'auto'}
  try:
    r = requests.get(u, params=p, timeout=30).json()
    d = pd.DataFrame(r['daily'])
    h = pd.DataFrame(r['hourly'])
    return {
      'temp_max_avg': round(d['temperature_2m_max'].mean(), 2),
      'temp_min_avg': round(d['temperature_2m_min'].mean(), 2),
      'temp_min_extreme': round(d['temperature_2m_min'].min(), 2),
      'precip_annual_avg': round(d['precipitation_sum'].sum() / 8, 2),
      'frost_days_avg': round((d['temperature_2m_min'] < 0).sum() / 8, 1),
      'evapotransp_avg': round(d['et0_fao_evapotranspiration'].mean(), 3),
      'soil_moisture_avg': round(h['soil_moisture_0_to_7cm'].mean(), 4),
      'soil_temp_avg': round(h['soil_temperature_0_to_7cm'].mean(), 2)}
  except:
    return None

res = []
for i, row in s.iterrows():
  name = row['state']
  dt = get_data(row['lat'], row['lon'])
  if dt:
    dt['state'] = name
    res.append(dt)
    print(name, 'ok')
  else:
    print(name, 'problem')
  time.sleep(1)
df_env = pd.DataFrame(res)
df_env.to_csv('env.csv', index=False)
df_env

California ok
Colorado ok
Florida ok
Georgia ok
Hawaii ok
Iowa problem
Kansas problem
Kentucky problem
Maryland problem
Nevada problem
New Mexico problem
New York problem
North Carolina problem
Ohio problem
Oklahoma problem
Oregon problem
Tennessee problem
Texas problem
Wisconsin problem


,temp_max_avg,temp_min_avg,temp_min_extreme,precip_annual_avg,frost_days_avg,evapotransp_avg,soil_moisture_avg,soil_temp_avg,state
0,24.05,12.72,-0.8,293.99,0.2,3.869,0.0673,22.01,California
1,17.58,3.32,-29.3,473.70,145.1,3.698,0.2102,12.49,Colorado
2,28.22,20.11,0.7,1347.50,0.0,3.888,0.1734,25.43,Florida
3,21.89,12.29,-13.6,1585.26,30.2,3.131,0.4063,18.04,Georgia
4,22.85,18.22,10.9,628.45,0.0,4.081,0.1871,22.80,Hawaii


We're going through the second time, collecting the states that didn't work out the first time.

In [ ]:
import pandas as pd
import requests
import time

if 'df_env' not in locals():
  df_env = pd.DataFrame()

done_list = []
if not df_env.empty:
  done_list = df_env['state'].tolist()

def get_data(lat, lon):
  u = 'https://archive-api.open-meteo.com/v1/archive'
  p = {
    'latitude': lat,
    'longitude': lon,
    'start_date': '2018-01-01',
    'end_date': '2022-12-31',
    'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum,et0_fao_evapotranspiration',
    'hourly': 'soil_moisture_0_to_7cm,soil_temperature_0_to_7cm',
    'timezone': 'auto'}
  try:
    r = requests.get(u, params=p, timeout=40).json()
    d = pd.DataFrame(r['daily'])
    h = pd.DataFrame(r['hourly'])
    return {
      'temp_max_avg': round(d['temperature_2m_max'].mean(), 2),
      'temp_min_avg': round(d['temperature_2m_min'].mean(), 2),
      'temp_min_extreme': round(d['temperature_2m_min'].min(), 2),
      'precip_annual_avg': round(d['precipitation_sum'].sum() / 5, 2),
      'frost_days_avg': round((d['temperature_2m_min'] < 0).sum() / 5, 1),
      'evapotransp_avg': round(d['et0_fao_evapotranspiration'].mean(), 3),
      'soil_moisture_avg': round(h['soil_moisture_0_to_7cm'].mean(), 4),
      'soil_temp_avg': round(h['soil_temperature_0_to_7cm'].mean(), 2)}
  except:
    return None

nw = []
names = s['state'].tolist()
lats = s['lat'].tolist()
lons = s['lon'].tolist()

for i in range(len(names)):
  n = names[i]
  if n in done_list:
    continue
  dt = get_data(lats[i], lons[i])
  if dt:
    dt['state'] = n
    nw.append(dt)
    print(n, 'ok')
  else:
    print(n, 'error')
  time.sleep(2)

if nw:
  df_new = pd.DataFrame(nw)
  df_env = pd.concat([df_env, df_new], ignore_index=True)
df_env.to_csv('env.csv', index=False)
df_env

Iowa error
Kansas ok
Kentucky ok
Maryland ok
Nevada ok
New Mexico ok
New York ok
North Carolina ok
Ohio ok
Oklahoma error
Oregon error
Tennessee error
Texas error
Wisconsin error


,temp_max_avg,temp_min_avg,temp_min_extreme,precip_annual_avg,frost_days_avg,evapotransp_avg,soil_moisture_avg,soil_temp_avg,state
0,24.05,12.72,-0.8,293.99,0.2,3.869,0.0673,22.01,California
1,17.58,3.32,-29.3,473.70,145.1,3.698,0.2102,12.49,Colorado
2,28.22,20.11,0.7,1347.50,0.0,3.888,0.1734,25.43,Florida
3,21.89,12.29,-13.6,1585.26,30.2,3.131,0.4063,18.04,Georgia
4,22.85,18.22,10.9,628.45,0.0,4.081,0.1871,22.80,Hawaii
5,18.25,8.57,-26.8,1260.60,92.6,3.104,0.3331,13.76,Kansas
6,19.41,9.22,-21.4,1378.32,77.2,2.918,0.3216,14.78,Kentucky
7,18.16,8.83,-16.8,1763.84,82.8,2.722,0.3992,14.14,Maryland
8,26.63,14.91,-4.1,107.86,7.2,5.855,0.0290,23.11,Nevada
9,21.45,8.21,-17.2,230.56,89.4,4.666,0.2837,17.21,New Mexico


Soil data
https://soilgrids.org/

In [ ]:
p = {'lon': -118.29,'lat': 34.03,'property': ['phh2o', 'soc', 'clay'],
    'depth': '0-5cm','value': 'mean'}
r = requests.get('https://rest.isric.org/soilgrids/v2.0/properties/query', params=p)
data = r.json()
data

{'type': 'Feature',
 'geometry': {'type': 'Point', 'coordinates': [-118.29, 34.03]},
 'properties': {'layers': [{'name': 'clay',
    'unit_measure': {'d_factor': 10,
     'mapped_units': 'g/kg',
     'target_units': '%',
     'uncertainty_unit': ''},
    'depths': [{'range': {'top_depth': 0,
       'bottom_depth': 5,
       'unit_depth': 'cm'},
      'label': '0-5cm',
      'values': {'mean': None}}]},
   {'name': 'phh2o',
    'unit_measure': {'d_factor': 10,
     'mapped_units': 'pH*10',
     'target_units': '-',
     'uncertainty_unit': ''},
    'depths': [{'range': {'top_depth': 0,
       'bottom_depth': 5,
       'unit_depth': 'cm'},
      'label': '0-5cm',
      'values': {'mean': None}}]},
   {'name': 'soc',
    'unit_measure': {'d_factor': 10,
     'mapped_units': 'dg/kg',
     'target_units': 'g/kg',
     'uncertainty_unit': ''},
    'depths': [{'range': {'top_depth': 0,
       'bottom_depth': 5,
       'unit_depth': 'cm'},
      'label': '0-5cm',
      'values': {'mean': None}

In [ ]:
import requests
points = [
    ('CA urban - LA', 34.03, -118.29),
    ('CA rural - Valley', 36.7, -119.7),
    ('CA Sacramento', 38.5, -121.5),
    ('TX rural', 31.0, -99.0),
    ('FL rural', 27.5, -81.5)]

for name, lat, lon in points:
  u = 'https://rest.isric.org/soilgrids/v2.0/properties/query'
  p = {'lon': lon,
        'lat': lat,
        'property': ['phh2o', 'clay'],
        'depth': '0-5cm',
        'value': 'mean'}
  r = requests.get(u, params=p).json()
  layers = r['properties']['layers']
  clay = layers[0]['depths'][0]['values']['mean']
  ph = layers[1]['depths'][0]['values']['mean']
  print(name, ':', 'pH =', ph, 'clay =', clay)

CA urban - LA : pH = None clay = None
CA rural - Valley : pH = 74 clay = 161
CA Sacramento : pH = 68 clay = 321
TX rural : pH = 70 clay = 392
FL rural : pH = None clay = None


The API works, but there is no data for this point, like Los Angeles is an urban area, SoilGrids does not cover it. The problem is that we take the average coordinates for the state that fall into the city. We need to try other coordinates, a little outside the city.

The data is available for rural points. The problem is that the average coordinates for the state fall into the cities. The solution is to simply use predefined rural coordinates for each state.

We just need urban data, plants in parks, on streets, in gardens. Thus we take the nearest rural point near the city like 5-20 km from the center. The soil there is similar to urban soil, just without asphalt on top.

In [ ]:
coor = {
  'California': (36.7, -119.7), 'Colorado': (39.0, -105.5),
  'Florida': (28.5, -82.5), 'Georgia': (32.5, -83.5),
  'Hawaii': (20.5, -156.5), 'Iowa': (42.0, -93.5),
  'Kansas': (38.5, -98.0), 'Kentucky': (37.5, -85.5),
  'Maryland': (39.0, -77.5), 'Nevada': (39.5, -117.0),
  'New Mexico': (34.5, -106.0), 'New York': (42.5, -75.5),
  'North Carolina': (35.5, -79.5), 'Ohio': (40.5, -82.5),
  'Oklahoma': (35.5, -97.5), 'Oregon': (44.0, -120.5),
  'Tennessee': (36.0, -86.5), 'Texas': (31.0, -99.0),
  'Wisconsin': (44.5, -89.5)}

props = ['phh2o', 'soc', 'clay', 'sand', 'silt', 'nitrogen', 'cec', 'bdod', 'cfvo', 'ocd', 'wv0010', 'wv0033', 'wv1500']
depths = ['0-5cm', '5-15cm', '15-30cm']

res = []
for st in coor:
  lat, lon = coor[st]
  print(st)
  row = {'state': st}
  for d in depths:
    try:
      u = 'https://rest.isric.org/soilgrids/v2.0/properties/query'
      p = {'lon': lon, 'lat': lat, 'property': props, 'depth': d, 'value': 'mean'}
      r = requests.get(u, params=p, timeout=30).json()
      layers = r['properties']['layers']

      sl = d.replace('-','_').replace('cm','')
      for l in layers:
        name = l['name']
        val = l['depths'][0]['values']['mean']
        factor = l['unit_measure']['d_factor']
        col = 'soil_' + name + '_' + sl
        if val is not None:
          row[col] = round(val/factor, 4)
        else:
          row[col] = None
      time.sleep(13)
    except:
      print('error', d)
  res.append(row)
  print('ok')

df_soil = pd.DataFrame(res)
df_soil.to_csv('soil.csv', index=False)
df_soil

California
ok
Colorado
ok
Florida
ok
Georgia
ok
Hawaii
ok
Iowa
ok
Kansas
ok
Kentucky
ok
Maryland
ok
Nevada
ok
New Mexico
ok
New York
ok
North Carolina
ok
Ohio
ok
Oklahoma
ok
Oregon
ok
Tennessee
ok
Texas
ok
Wisconsin
ok


,state,soil_bdod_0_5,soil_cec_0_5,soil_cfvo_0_5,soil_clay_0_5,soil_nitrogen_0_5,soil_ocd_0_5,soil_phh2o_0_5,soil_sand_0_5,soil_silt_0_5,...,soil_clay_15_30,soil_nitrogen_15_30,soil_ocd_15_30,soil_phh2o_15_30,soil_sand_15_30,soil_silt_15_30,soil_soc_15_30,soil_wv0010_15_30,soil_wv0033_15_30,soil_wv1500_15_30
0,California,1.55,17.4,2.5,16.1,4.79,27.9,7.4,44.9,39.0,...,16.1,0.95,15.3,7.5,44.4,39.4,7.2,30.3,24.0,13.5
1,Colorado,1.12,17.5,17.8,17.3,5.52,46.1,6.0,52.0,30.6,...,18.7,1.32,16.3,5.9,54.5,26.8,15.7,30.8,23.8,12.2
2,Florida,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Georgia,1.35,15.1,2.5,22.2,2.63,47.9,5.2,37.1,40.7,...,24.7,0.85,16.4,5.2,36.1,39.2,13.9,31.5,28.1,17.3
4,Hawaii,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Iowa,1.49,24.1,4.2,25.9,2.85,32.5,6.6,38.8,35.3,...,27.4,2.22,37.0,6.5,37.9,34.6,19.0,37.8,33.4,17.7
6,Kansas,1.40,18.1,1.7,25.1,2.74,33.1,6.7,38.7,36.2,...,28.2,1.04,16.8,6.9,36.3,35.6,10.2,34.4,29.4,15.4
7,Kentucky,1.30,16.9,5.8,20.4,3.50,46.3,5.5,15.3,64.2,...,28.1,1.05,14.1,5.4,12.8,59.1,9.4,39.0,34.5,17.4
8,Maryland,1.34,27.7,6.2,23.4,4.77,40.6,5.7,27.8,48.8,...,27.9,0.88,15.8,5.8,26.8,45.4,9.3,35.4,31.8,14.0
9,Nevada,1.39,18.1,11.7,17.3,1.27,20.4,7.3,43.1,39.7,...,22.8,0.83,12.6,7.5,40.0,37.2,9.6,35.5,28.4,16.7


In [ ]:
es = df_soil[df_soil['soil_phh2o_0_5'].isna()]['state'].tolist()
print(f'empty {es}')

empty ['Florida', 'Hawaii', 'Oklahoma', 'Tennessee']


In [ ]:
fix_coor = {
  'Florida': (29.5, -82.0),
  'Hawaii': (19.8, -155.5),
  'Oklahoma': (36.0, -96.5),
  'Tennessee': (35.8, -87.0)}
fix_res = []
for st in fix_coor:
  lat, lon = fix_coor[st]
  print(st)
  row = {'state': st}

  for d in depths:
    try:
      u = 'https://rest.isric.org/soilgrids/v2.0/properties/query'
      p = {'lon': lon, 'lat': lat, 'property': props, 'depth': d, 'value': 'mean'}
      r = requests.get(u, params=p, timeout=30).json()
      layers = r['properties']['layers']
      label = d.replace('-','_').replace('cm','')

      for l in layers:
        n = l['name']
        v = l['depths'][0]['values']['mean']
        f = l['unit_measure']['d_factor']
        col = 'soil_' + n + '_' + label
        if v is not None:
          row[col] = round(v / f, 4)

      print(d, 'ok')
      time.sleep(13)
    except:
      print(d, 'error')
  fix_res.append(row)
df_fix = pd.DataFrame(fix_res)
if 'df_soil' in locals():
  df_soil = df_soil[df_soil['state'].isin(fix_coor.keys()) == False]
  df_soil = pd.concat([df_soil, df_fix], ignore_index=True)
else:
  df_soil = df_fix
df_soil

Florida
0-5cm ok
5-15cm ok
15-30cm ok
Hawaii
0-5cm ok
5-15cm ok
15-30cm ok
Oklahoma
0-5cm ok
5-15cm ok
15-30cm ok
Tennessee
0-5cm ok
5-15cm ok
15-30cm ok


,state,soil_bdod_0_5,soil_cec_0_5,soil_cfvo_0_5,soil_clay_0_5,soil_nitrogen_0_5,soil_ocd_0_5,soil_phh2o_0_5,soil_sand_0_5,soil_silt_0_5,...,soil_clay_15_30,soil_nitrogen_15_30,soil_ocd_15_30,soil_phh2o_15_30,soil_sand_15_30,soil_silt_15_30,soil_soc_15_30,soil_wv0010_15_30,soil_wv0033_15_30,soil_wv1500_15_30
0,California,1.55,17.4,2.5,16.1,4.79,27.9,7.4,44.9,39.0,...,16.1,0.95,15.3,7.5,44.4,39.4,7.2,30.3,24.0,13.5
1,Colorado,1.12,17.5,17.8,17.3,5.52,46.1,6.0,52.0,30.6,...,18.7,1.32,16.3,5.9,54.5,26.8,15.7,30.8,23.8,12.2
2,Georgia,1.35,15.1,2.5,22.2,2.63,47.9,5.2,37.1,40.7,...,24.7,0.85,16.4,5.2,36.1,39.2,13.9,31.5,28.1,17.3
3,Iowa,1.49,24.1,4.2,25.9,2.85,32.5,6.6,38.8,35.3,...,27.4,2.22,37.0,6.5,37.9,34.6,19.0,37.8,33.4,17.7
4,Kansas,1.40,18.1,1.7,25.1,2.74,33.1,6.7,38.7,36.2,...,28.2,1.04,16.8,6.9,36.3,35.6,10.2,34.4,29.4,15.4
5,Kentucky,1.30,16.9,5.8,20.4,3.50,46.3,5.5,15.3,64.2,...,28.1,1.05,14.1,5.4,12.8,59.1,9.4,39.0,34.5,17.4
6,Maryland,1.34,27.7,6.2,23.4,4.77,40.6,5.7,27.8,48.8,...,27.9,0.88,15.8,5.8,26.8,45.4,9.3,35.4,31.8,14.0
7,Nevada,1.39,18.1,11.7,17.3,1.27,20.4,7.3,43.1,39.7,...,22.8,0.83,12.6,7.5,40.0,37.2,9.6,35.5,28.4,16.7
8,New Mexico,1.49,12.6,7.6,21.4,1.04,16.2,7.7,53.5,25.1,...,23.8,0.79,11.5,7.9,51.9,24.3,6.7,31.4,27.8,14.0
9,New York,1.09,45.8,10.8,18.4,9.62,56.9,4.7,29.1,52.5,...,18.2,1.68,21.4,5.0,30.1,51.7,26.0,38.4,33.4,13.1


In [ ]:
df_soil.columns

Index(['state', 'soil_bdod_0_5', 'soil_cec_0_5', 'soil_cfvo_0_5',
       'soil_clay_0_5', 'soil_nitrogen_0_5', 'soil_ocd_0_5', 'soil_phh2o_0_5',
       'soil_sand_0_5', 'soil_silt_0_5', 'soil_soc_0_5', 'soil_wv0010_0_5',
       'soil_wv0033_0_5', 'soil_wv1500_0_5', 'soil_bdod_5_15', 'soil_cec_5_15',
       'soil_cfvo_5_15', 'soil_clay_5_15', 'soil_nitrogen_5_15',
       'soil_ocd_5_15', 'soil_phh2o_5_15', 'soil_sand_5_15', 'soil_silt_5_15',
       'soil_soc_5_15', 'soil_wv0010_5_15', 'soil_wv0033_5_15',
       'soil_wv1500_5_15', 'soil_bdod_15_30', 'soil_cec_15_30',
       'soil_cfvo_15_30', 'soil_clay_15_30', 'soil_nitrogen_15_30',
       'soil_ocd_15_30', 'soil_phh2o_15_30', 'soil_sand_15_30',
       'soil_silt_15_30', 'soil_soc_15_30', 'soil_wv0010_15_30',
       'soil_wv0033_15_30', 'soil_wv1500_15_30'],
      dtype='object')

**Block 2**. We will collect data on the styles of gardens that correspond to the plants from the site.

 https://plants.ces.ncsu.edu/find_a_plant/

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/merged_city_result-2.csv')
df.shape

/tmp/ipykernel_41731/2757962669.py:2: DtypeWarning: Columns (5,6,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/merged_city_result-2.csv')


(1013701, 61)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import requests
from bs4 import BeautifulSoup
import time

We collect the exact matches of scientific names and pages on the site, add them to the column df['themes']

In [ ]:
def get_themes(u):
  try:
    r = requests.get(u, timeout=10).text
    s = BeautifulSoup(r, 'html.parser')
    t = s.find('dt', string=lambda x: x and 'Landscape Theme:' in x)
    if not t:
      return None
    res = []
    for n in t.find_next_siblings():
      if n.name == 'dt': break
      if n.name == 'dd': res.append(n.get_text(strip=True))
    return res if res else None
  except:
    return None

names = df['sci_name_clean'].dropna().unique()[500:]
for n in names:
  slugs = [n.lower().replace(' ', '-'), n.split(' ')[0].lower()]
  for s in slugs:
    url = 'https://plants.ces.ncsu.edu/plants/' + s + '/'
    data = get_themes(url)
    if data:
      idx = df[df['sci_name_clean'] == n].index
      df.loc[idx, 'themes'] = str(data)
      break
  print(n, data)
  time.sleep(1)

juglans None
catalpa None
abies None
euptelea pleiosperma None
clerodendrum None
styrax None
trachycarpus None
chamaecyparis ['Asian Garden', 'English Garden', 'Water Garden', 'Winter Garden']
castanea None
aralia None
laburnum None
cotinus ['Drought Tolerant Garden', 'Pollinator Garden']
betula None
carpinus None
salix ['Butterfly Garden', 'Pollinator Garden']
corylus None
lagerstroemia indica ['Asian Garden', 'Butterfly Garden', "Children's Garden", 'Cottage Garden', 'Pollinator Garden', 'Winter Garden']
gymnocladus dioica None
olea None
rhamnus purshiana ['Native Garden']
eucalyptus ['Cutting Garden', 'Drought Tolerant Garden', 'Winter Garden']
juniperus None
arbutus menziesii ['Drought Tolerant Garden', 'Pollinator Garden', 'Winter Garden']
larix None
araucaria araucana ['Asian Garden', 'Winter Garden']
euonymus ['Asian Garden', 'Native Garden', 'Shade Garden']
camellia None
azara None
alnus None
heptacodium miconioides ['Cottage Garden', 'Pollinator Garden', 'Winter Garden']
arbut

In [ ]:
print(f'пропусков {df['themes'].isna().sum()} из {len(df)}')

пропусков 84080 из 1013701


In [ ]:
names = df[df['themes'].isna()]['sci_name_clean'].dropna().unique()

for n in names:
  g = n.split()[0].lower()
  u = 'https://ncsu.edu' + g + '/'
  try:
    r = requests.get(u, timeout=10).text
    s = BeautifulSoup(r, 'html.parser')
    link = ''
    for a in s.find_all('a', href=True):
      if '/plants/' in a['href'] and a['href'].count('/') > 2:
        link = 'https://ncsu.edu' + a['href']
        break

    if link:
      r2 = requests.get(link, timeout=10).text
      s2 = BeautifulSoup(r2, 'html.parser')
      t = s2.find('dt', string=lambda x: x and 'Landscape Theme:' in x)
      if t:
        vals = []
        for v in t.find_next_siblings():
          if v.name == 'dt': break
          if v.name == 'dd': vals.append(v.get_text(strip=True))
        df.loc[df['sci_name_clean'] == n, 'themes'] = '; '.join(vals)
        print(n, 'ok')
      else:
        print(n, 'no themes')
    else:
      print(n, 'not found')
  except:
    print(n, 'error')

  time.sleep(1)
df['themes'].isna().sum()

pinus eldarica error
fraxinus error
pinus error
fraxinus excelsior error
fraxinus quadrangulata error
pinus resinosa error
fraxinus ornus error
pinus strobiformis error
pinus leucodermis error
polyscias fruticosa error
polyscias guilfoylei error
fraxinus velutina error
fraxinus pennsylvanica 'patmore' error
fraxinus mandshurica error
fraxinus uhdei error
pinus canariensis error
pinus halepensis error
pinus radiata error
pinus patula error
pinus torreyana error
pinus brutia error
pinus coulteri error
pinus roxburghii error
pinus sabiniana error
pinus muricata error
abutilon pictum error
fraxinus angustifolia error


np.int64(84080)

In [ ]:
import re
res_set = set()
for i in df['themes'].dropna():
    txt = re.sub(r"[\[\];]", ',', str(i))
    ch = txt.split(',')
    for p in ch:
        p = p.strip().strip("'").strip('"')
        if p != '':
            res_set.add(p)

res = sorted(list(res_set))
print(res)

['Asian Garden', 'Butterfly Garden', "Children's Garden", 'Cottage Garden', 'Cutting Garden', 'Drought Tolerant Garden', 'Edible Garden', 'English Garden', 'Fairy Garden', 'Garden for the Blind', 'Native Garden', 'Nighttime Garden', 'Pollinator Garden', 'Rain Garden', 'Rock Garden', 'Shade Garden', 'Water Garden', 'Winter Garden']


These are all our final garden styles.

In [ ]:
df.to_csv('my_plants_data.csv', index=False)